# Data Transformation — PySpark + Delta Lake version (Databricks)

Spark/Delta-native counterpart to `data_transformation_reference.ipynb`. Same
concepts (flattening, normalization, schema drift, cleaning, dedupe, validation)
but expressed as DataFrame transformations over whole datasets, writing results
to managed Delta tables instead of returning Python lists/dicts.

Run on a Databricks cluster (uses the pre-attached `spark`/`dbutils`), or locally
with `pyspark` + `delta-spark` installed — the setup cell below detects which
environment it's in.

## Setup

In [4]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, ArrayType, StringType, IntegerType

try:
    spark                                    # already provided by the Databricks notebook runtime
except NameError:
    from delta import configure_spark_with_delta_pip
    builder = (SparkSession.builder
               .appName("data_transformation_reference")
               .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
               .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

try:
    dbutils                                  # already provided on Databricks
except NameError:
    dbutils = None

In [5]:
# Target schema for the Delta tables this notebook writes. On Databricks with
# Unity Catalog, set `catalog` to a real catalog name; leave it blank to fall
# back to a plain `schema.table` (works on hive_metastore and locally too).
if dbutils is not None:
    dbutils.widgets.text("catalog", "", "Unity Catalog (blank = hive_metastore)")
    dbutils.widgets.text("schema", "devrev_ref", "Schema name")
    CATALOG = dbutils.widgets.get("catalog").strip()
    SCHEMA = dbutils.widgets.get("schema").strip() or "devrev_ref"
else:
    CATALOG = ""
    SCHEMA = "devrev_ref"

SCHEMA_FQN = f"{CATALOG}.{SCHEMA}" if CATALOG else SCHEMA

def qualified(table):
    return f"{SCHEMA_FQN}.{table}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQN}")
print("Delta tables will be written under:", SCHEMA_FQN)

Box(children=(Label(value='Unity Catalog (blank = hive_metastore)'), Text(value='')))

Box(children=(Label(value='Schema name'), Text(value='devrev_ref')))

Delta tables will be written under: devrev_ref


## 1. Flattening

The Python reference walks one nested `dict` at a time. On Spark you flatten the
whole DataFrame's *schema* at once: expand `struct` columns into `parent.child`
columns, and `explode` `array<struct>` columns (arrays of scalars are left alone,
same rule as the Python version).

In [7]:
def _exact_col(name):
    """F.col("a.b") parses the dot as nested-field access. Once flattening has put a
    literal "." into a column name, the only way to reference that exact top-level
    column is to backtick-quote the whole name."""
    return F.col("`" + name.replace("`", "``") + "`")


def flatten_df(df, sep="."):
    """Flatten struct columns to dot-notation; explode array<struct> columns.
    Arrays of scalars are left untouched (matches the plain-Python reference)."""
    while True:
        complex_fields = {
            f.name: f.dataType for f in df.schema.fields
            if isinstance(f.dataType, StructType)
            or (isinstance(f.dataType, ArrayType) and isinstance(f.dataType.elementType, StructType))
        }
        if not complex_fields:
            return df
        name, dtype = next(iter(complex_fields.items()))
        if isinstance(dtype, ArrayType):
            df = df.withColumn(name, F.explode_outer(_exact_col(name)))  # array<struct> -> one row per element
            dtype = dtype.elementType
        expanded = [_exact_col(name).getField(c).alias(f"{name}{sep}{c}") for c in dtype.fieldNames()]
        print(f"expanded is {expanded}")
        # a plain string in select() is parsed the same dot-splitting way as F.col(str) --
        # any already-flattened column here may itself contain a literal "." (e.g. from a
        # previous pass), so it needs the same backtick-quoted exact-name reference too.
        others = [_exact_col(c) for c in df.columns if c != name]
        print(f"others are {others}")
        df = df.select(*others, *expanded)

## 1b. Normalize tickets into FK-linked Delta tables

The Python version explodes *one* ticket dict into 4 Python lists. The Spark
version explodes the `conversations` / `messages` / `attachments` arrays across
*every* ticket in the DataFrame at once, producing the 4 star-schema tables in a
single pass — this is the real replacement for `normalize_ticket`.

In [8]:
def normalize_tickets(df):
    """df: one row per ticket, with nested `conversations` array.
    Returns (tickets, conversations, messages, attachments) DataFrames with FKs."""
    tickets = df.select("id", "subject", "status")

    conv = (df.select(F.col("id").alias("ticket_id"), F.explode("conversations").alias("c"))
              .select("ticket_id", F.col("c.id").alias("id"), F.col("c.channel").alias("channel")))

    msg_raw = (df.select(F.col("id").alias("ticket_id"), F.explode("conversations").alias("c"))
                 .select(F.col("c.id").alias("conversation_id"), F.explode("c.messages").alias("m")))
    messages = msg_raw.select(
        "conversation_id", F.col("m.id").alias("id"),
        F.col("m.author").alias("author"), F.col("m.body").alias("body"))

    attachments = (msg_raw.select(F.col("m.id").alias("message_id"), F.explode_outer("m.attachments").alias("a"))
                          .filter(F.col("a").isNotNull())
                          .select("message_id", F.col("a.id").alias("id"),
                                  F.col("a.filename").alias("filename"), F.col("a.url").alias("url")))
    return tickets, conv, messages, attachments

## 2. Schema handling

A Spark column has one static type, so the Python `coerce_assignee` (branch on
`isinstance`) isn't idiomatic here. Instead: normalize each differently-shaped
*source* to the same target struct, then `unionByName`. Renamed-key tolerance
becomes `F.coalesce` across candidate columns; "missing vs explicit null" is a
per-row null check once the column exists (Spark already tells missing columns
apart from null values at the schema level).

In [9]:
def normalize_assignee_v1(df):
    """v1 shape: assignee is a plain string -> lift it into struct{id,name}."""
    return df.withColumn("assignee", F.struct(F.lit(None).cast("int").alias("id"),
                                               F.col("assignee").alias("name")))

def normalize_assignee_v2(df):
    """v2 shape: assignee is already struct{id,name} -> pass through."""
    return df.withColumn("assignee", F.struct(F.col("assignee.id").alias("id"),
                                               F.col("assignee.name").alias("name")))

FIELD_MAP = {
    "id":      ["id", "ticket_id", "uuid"],
    "subject": ["subject", "title", "summary"],
    "created": ["created_at", "createdAt", "created"],
}

def get_field_expr(df, aliases):
    """First present, non-null column among candidate names (handles renamed keys)."""
    cols = [F.col(c) for c in aliases if c in df.columns]
    return F.coalesce(*cols) if cols else F.lit(None)

def transform_df(df):
    return df.select(*[get_field_expr(df, aliases).alias(out) for out, aliases in FIELD_MAP.items()])

def resolve_col(df, key, on_missing, on_null):
    """Missing column vs explicit null are different -> return the right default for each."""
    if key not in df.columns:
        return df.withColumn(key, F.lit(on_missing))
    return df.withColumn(key, F.when(F.col(key).isNull(), F.lit(on_null)).otherwise(F.col(key)))

## 3. Data cleaning

- `to_iso8601` -> native `F.to_timestamp` tried across each candidate format and
  `F.coalesce`d together, instead of a Python `try/except` loop per record.
- `business_key_hash` -> native `F.sha2`, computed for the whole column at once.
- `dedupe` -> `Window` + `row_number()` (or a Delta `MERGE INTO` if you're
  deduping incrementally against a target table rather than within one batch).
- `validate` -> split into a *good* and *bad* DataFrame (the standard
  "quarantine" pattern) rather than a per-record list of error strings. For
  real type-safety, prefer enforcing a `StructType` on read plus Databricks'
  `rescuedDataColumn` option, which automatically captures rows that don't
  match the expected schema instead of silently dropping or crashing.

In [10]:
_DATE_FORMATS = ["yyyy-MM-dd HH:mm:ss", "yyyy-MM-dd", "MM/dd/yyyy", "dd-MMM-yyyy"]

def to_iso8601_col(c):
    """Best-effort parse across several formats -> canonical UTC timestamp column."""
    numeric = c.cast("double")
    from_epoch = F.to_timestamp(F.when(numeric > 1e12, numeric / 1000).otherwise(numeric))
    iso = F.to_timestamp(F.regexp_replace(c, "Z$", "+00:00"))
    from_formats = [F.to_timestamp(c, fmt) for fmt in _DATE_FORMATS]
    return F.coalesce(F.when(numeric.isNotNull(), from_epoch), iso, *from_formats)


def business_key_hash_col(keys):
    """Stable fingerprint from chosen business keys (normalized for consistency)."""
    parts = [F.lower(F.trim(F.coalesce(F.col(k).cast("string"), F.lit("")))) for k in keys]
    return F.sha2(F.concat_ws("|", *parts), 256)


def dedupe_df(df, keys):
    """First occurrence (by input order) wins, same semantics as the Python version."""
    keyed = df.withColumn("_bkey", business_key_hash_col(keys)) \
              .withColumn("_ord", F.monotonically_increasing_id())
    w = Window.partitionBy("_bkey").orderBy("_ord")
    return (keyed.withColumn("_rn", F.row_number().over(w))
                 .filter(F.col("_rn") == 1)
                 .drop("_bkey", "_ord", "_rn"))


def validate_df(df, required_fields):
    """Split into (good, bad) DataFrames; `bad` carries an `_errors` array column."""
    reasons = [F.when(F.col(f).isNull(), F.lit(f"missing required field '{f}'"))
               if f in df.columns else F.lit(f"missing required field '{f}'")
               for f in required_fields]
    tagged = df.withColumn("_errors", F.array_compact(F.array(*reasons)))
    good = tagged.filter(F.size("_errors") == 0).drop("_errors")
    bad = tagged.filter(F.size("_errors") > 0)
    return good, bad

## Demo / self-test

### 1. Flatten to dot-notation

In [11]:
payload_df = spark.createDataFrame([
    ("Ada", "Pune", "411001", ["urgent", "billing"]),
], "name string, city string, zip string, tags array<string>") \
    .select(F.struct(
        F.col("name"),
        F.struct(F.col("city"), F.col("zip")).alias("address"),
    ).alias("customer"), F.col("tags"))

flat = flatten_df(payload_df)
flat.show(truncate=False)
row = flat.first()
assert row["customer.address.city"] == "Pune"
assert row["tags"] == ["urgent", "billing"]   # array of scalars kept as one value

expanded is [Column<'`customer`['name'] AS customer.name'>, Column<'`customer`['address'] AS customer.address'>]
others are [Column<'`tags`'>]
expanded is [Column<'`customer.address`['city'] AS customer.address.city'>, Column<'`customer.address`['zip'] AS customer.address.zip'>]
others are [Column<'`tags`'>, Column<'`customer.name`'>]
+-----------------+-------------+---------------------+--------------------+
|tags             |customer.name|customer.address.city|customer.address.zip|
+-----------------+-------------+---------------------+--------------------+
|[urgent, billing]|Ada          |Pune                 |411001              |
+-----------------+-------------+---------------------+--------------------+



In [12]:
display(payload_df)

,customer,tags
0,"{'name': 'Ada', 'address': {'city': 'Pune', 'zip': '411001'}}","[urgent, billing]"


In [13]:
payload_df.printSchema()

root
 |-- customer: struct (nullable = false)
 |    |-- name: string (nullable = true)
 |    |-- address: struct (nullable = false)
 |    |    |-- city: string (nullable = true)
 |    |    |-- zip: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)



### 1b. Normalize tickets -> FK-linked Delta tables

In [14]:
ticket_schema = """
    id string, subject string, status string,
    conversations array<struct<
        id: string, channel: string,
        messages: array<struct<
            id: string, author: string, body: string,
            attachments: array<struct<id: string, filename: string, url: string>>
        >>
    >>
"""
tickets_raw = spark.createDataFrame([{
    "id": "T1", "subject": "Cannot log in", "status": "open",
    "conversations": [{
        "id": "C1", "channel": "email",
        "messages": [
            {"id": "M1", "author": "user", "body": "help!",
             "attachments": [{"id": "A1", "filename": "err.png", "url": "http://x/err.png"}]},
            {"id": "M2", "author": "agent", "body": "try reset", "attachments": []},
        ],
    }],
}], ticket_schema)

tickets, conversations, messages, attachments = normalize_tickets(tickets_raw)
for name, df in [("tickets", tickets), ("conversations", conversations),
                  ("messages", messages), ("attachments", attachments)]:
    df.write.format("delta").mode("overwrite").saveAsTable(qualified(name))
    print(f"  {name}: {spark.table(qualified(name)).count()} row(s) -> {qualified(name)}")

assert spark.table(qualified("conversations")).first()["ticket_id"] == "T1"     # FK check
assert spark.table(qualified("messages")).first()["conversation_id"] == "C1"
assert spark.table(qualified("attachments")).first()["message_id"] == "M1"

  tickets: 1 row(s) -> devrev_ref.tickets


  conversations: 1 row(s) -> devrev_ref.conversations


  messages: 2 row(s) -> devrev_ref.messages


  attachments: 1 row(s) -> devrev_ref.attachments


### 2. Schema drift: assignee string (v1) vs struct (v2) -> unioned

In [15]:
v1 = spark.createDataFrame([("T9", "Ada")], "id string, assignee string")
v2 = spark.createDataFrame([("T7", (7, "Ada"))], "id string, assignee struct<id:int,name:string>")

# combined = normalize_assignee_v1(v1).unionByName(normalize_assignee_v2(v2))
# combined.show(truncate=False)
# rows = {r["id"]: r["assignee"] for r in combined.collect()}
# assert rows["T9"]["name"] == "Ada" and rows["T9"]["id"] is None
# assert rows["T7"]["id"] == 7

In [18]:
display(v1)

,id,assignee
0,T9,Ada


In [17]:
display(normalize_assignee_v1(v1))

,id,assignee
0,T9,"{'id': None, 'name': 'Ada'}"


In [19]:
display(v2)

,id,assignee
0,T7,"{'id': 7, 'name': 'Ada'}"


In [20]:
display(normalize_assignee_v2(v2))

,id,assignee
0,T7,"{'id': 7, 'name': 'Ada'}"


### 2b. Tolerant field mapping (renamed keys)

In [ ]:
renamed = spark.createDataFrame([("T9", "Bug", "2024-01-01")], "ticket_id string, title string, createdAt string")
mapped = transform_df(renamed)
mapped.show(truncate=False)
assert mapped.first()["id"] == "T9"

+---+-------+----------+
|id |subject|created   |
+---+-------+----------+
|T9 |Bug    |2024-01-01|
+---+-------+----------+



: 

### 2c. Missing vs explicit null

In [ ]:
missing_phone = spark.createDataFrame([("T1",)], "id string")
null_phone = spark.createDataFrame([("T1", None)], "id string, phone string")

r1 = resolve_col(missing_phone, "phone", on_missing="<keep>", on_null=None).first()
r2 = resolve_col(null_phone, "phone", on_missing="<keep>", on_null=None).first()
print("  missing phone ->", r1["phone"])
print("  null phone    ->", r2["phone"])
assert r1["phone"] == "<keep>"
assert r2["phone"] is None

### 3. Dates -> ISO 8601 (UTC)

In [ ]:
dates = spark.createDataFrame(
    [("2024-03-01",), ("03/01/2024",), ("1709294400",), ("2024-03-01T12:00:00Z",)],
    "raw string")
parsed = dates.withColumn("parsed", to_iso8601_col(F.col("raw")))
parsed.show(truncate=False)
rows = {r["raw"]: r["parsed"] for r in parsed.collect()}
assert str(rows["2024-03-01"]).startswith("2024-03-01 00:00:00")
assert str(rows["2024-03-01T12:00:00Z"]).startswith("2024-03-01 12:00:00")

### 3b. Dedupe on business keys

In [ ]:
recs = spark.createDataFrame([
    ("A@x.com ", "Ada"),
    ("a@x.com", "Ada A"),   # same person, messy email
    ("b@x.com", "Bo"),
], "email string, name string")

unique = dedupe_df(recs, keys=["email"])
print(f"  {recs.count()} in -> {unique.count()} unique")
assert unique.count() == 2

### 3c. Validate -> good / bad (quarantine) split

In [ ]:
candidates = spark.createDataFrame([
    ("x", 3),
    (None, 3),
], "id string, count int")

good, bad = validate_df(candidates, required_fields=["id", "count"])
print("  good:", good.count())
print("  bad :", bad.count())
bad.show(truncate=False)
assert good.count() == 1
assert bad.count() == 1

print("\nALL CHECKS PASSED")